In [1]:
import sqlite3
import pandas as pd

con = sqlite3.connect('database.sqlite')

In [2]:
CATEGORICAL_COLUMNS = [
    'attackComplexity',
    'attackVector',
    'availabilityImpact',
    'baseSeverity',
    'confidentialityImpact',
    'integrityImpact',
    'privilegesRequired',
    'scope',
    'userInteraction'
]

In [3]:
dfs = []
for year in [2022, 2023, 2024, 2025]:
    try:
        df_year = pd.read_sql_query(f"SELECT * FROM YEAR_{year}", con)
        df_year['source_year'] = year  # Keep track of the year group
        dfs.append(df_year)
    except Exception as e:
        print(f"Warning: Could not read table YEAR_{year}: {e}")

db = pd.concat(dfs, ignore_index=True)
db.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 121429 entries, 0 to 121428
Data columns (total 17 columns):
 #   Column                 Non-Null Count   Dtype  
---  ------                 --------------   -----  
 0   cveId                  121429 non-null  object 
 1   dataVersion            121429 non-null  float64
 2   datePublished          121429 non-null  object 
 3   product                109806 non-null  object 
 4   vendor                 106581 non-null  object 
 5   cweId                  111264 non-null  object 
 6   attackComplexity       121429 non-null  object 
 7   attackVector           121429 non-null  object 
 8   availabilityImpact     121429 non-null  object 
 9   baseScore              121429 non-null  float64
 10  baseSeverity           121429 non-null  object 
 11  confidentialityImpact  121429 non-null  object 
 12  integrityImpact        121429 non-null  object 
 13  privilegesRequired     121429 non-null  object 
 14  scope                  121429 non-nu

In [4]:
def display_yearly_breakdown(df, column_name):
    # crosstab puts the feature on rows and source_year on columns
    breakdown = pd.crosstab(df[column_name], df['source_year']).fillna(0).astype(int)

    # Replace top left corner with the column name for clarity
    breakdown.index.name = None
    breakdown.columns.name = column_name
            
    # Sort columns chronologically
    breakdown = breakdown[sorted(breakdown.columns)]
    display(breakdown)

In [5]:
for col in CATEGORICAL_COLUMNS:
    if col in db.columns:
        display_yearly_breakdown(db, col)

attackComplexity,2022,2023,2024,2025
High,2180,2656,3496,4293
Low,15315,22119,33562,37780
Medium,2,10,9,7


attackVector,2022,2023,2024,2025
Adjacent,621,1143,1413,1244
Local,5415,5979,7128,7327
Network,11265,17440,28163,33101
Physical,196,223,363,408


availabilityImpact,2022,2023,2024,2025
Complete,1,83,378,735
High,9387,11043,13705,12640
Low,1931,3549,5424,6775
None,6145,9167,15852,18078
Partial,33,943,1708,3852


baseSeverity,2022,2023,2024,2025
Critical,1860,2500,3772,3800
High,6760,8967,13225,16406
Low,877,1299,1770,2414
Medium,7998,12014,18288,19420
None,2,5,12,40


confidentialityImpact,2022,2023,2024,2025
Complete,1,46,352,701
High,9029,11568,14996,14327
Low,3954,6021,11150,11286
None,4485,6185,8773,11836
Partial,28,965,1796,3930


integrityImpact,2022,2023,2024,2025
Complete,1,44,349,701
High,7820,9944,12441,11609
Low,4106,6672,11681,12300
None,5521,6806,10234,12882
Partial,49,1319,2362,4588


privilegesRequired,2022,2023,2024,2025
High,2495,3367,3736,3548
Low,6242,7876,12886,13213
None,8760,13542,20445,25319


scope,2022,2023,2024,2025
Changed,1,29,134,192
Complete,3278,5452,9583,9166
None,1,4,46,74
Partial,0,0,28,56
Unchanged,14217,19300,27276,32592


userInteraction,2022,2023,2024,2025
Active,5,21,320,796
None,11939,16740,25781,30141
Passive,9,43,312,850
Required,5544,7981,10654,10293


In [6]:
# Only numeric columns are baseScore
score_breakdown = pd.crosstab(db['baseScore'], db['source_year']).fillna(0).astype(int)

# Replace top left corner with the column name for clarity
score_breakdown.index.name = None
score_breakdown.columns.name = 'baseScore'

display(score_breakdown[sorted(score_breakdown.columns)])

baseScore,2022,2023,2024,2025
0.0,2,5,12,40
0.1,0,0,2,0
0.3,0,0,0,1
0.5,0,0,0,2
0.6,0,0,0,12
...,...,...,...,...
9.6,85,95,116,78
9.7,1,21,13,22
9.8,1224,1668,2151,1798
9.9,84,95,144,161
